# Predicting Realized Volatility with Machine Learning
### A Data Science Project — Optiver Dataset
**Hayden Wong** | Python, DuckDB/SQL, LightGBM, SHAP | https://github.com/haydenwongg647/optiver-realized-volatility

## 1. Introduction

Financial markets move constantly, and knowing how much a stock's price is likely to move in the near future, its volatility, is one of the most fundamental problems in trading and risk management. This project tackles that problem directly using the Optiver Realized Volatility Prediction dataset, real order book and trade data released by Optiver, a global market maker, covering 112 stocks across thousands of 10-minute trading windows.

The task is to predict the realized volatility of a stock over the 10 minutes immediately following a given window, using only the order book (bid/ask prices and sizes) and trade data observed during that window. Unlike the credit risk problem in my previous project, which involved classifying a binary outcome from static applicant data, this is a regression problem built on high-frequency, time-ordered market microstructure data. The central challenge shifts from feature engineering across relational tables to extracting signal from noisy, rapidly changing price and order flow.

This project also marks a deliberate shift in tooling: rather than R, I built this pipeline in Python, using DuckDB and SQL for feature engineering directly against the raw order book data before handing off to Python for modelling and interpretability. That split, SQL for aggregation, Python for modelling, reflects a workflow I wanted to practise directly, since it mirrors how these problems are approached in industry.

This dataset was chosen specifically because Optiver is a market maker, so the problem it poses sits close to the actual commercial motivation behind the data. This isn't a synthetic or toy problem; it's the kind of question a trading firm's systems answer continuously in production. 

## 2. Data Overview and Cleaning 

### 2.1 Data Overview 

The data comes from the Optiver Realized Volatility Prediction competition on Kaggle, and consists of real, anonymised order book and trade data across 112 stocks, split into thousands of independent 10-minute trading windows identified by stock_id and time_id.

Two raw data sources are provided per stock:

Order book data (book_train.parquet) — a snapshot of the top two price levels on both sides of the book, recorded every time the book changes within a window:

seconds_in_bucket — time elapsed within the 10-minute window
bid_price1/2, ask_price1/2 — best and second-best bid/ask prices
bid_size1/2, ask_size1/2 — volume available at each price level

Trade data (trade_train.parquet) — actual executed trades within the same window:

price, size, order_count — the price, volume, and number of orders behind each trade

train.csv provides the target: the realized volatility of the following 10-minute window for each stock_id/time_id pair, meaning the task is genuinely predictive, not descriptive. The model has to forecast what happens next using only what's observable in the current window.

This structure creates a different kind of challenge to Home Credit. Rather than joining relational tables that describe an applicant's static history, the raw unit of data here is a rapidly changing time series within each window, and the feature engineering problem is about compressing that irregular, high-frequency signal into a fixed set of summary statistics per stock_id/time_id before any model can be trained.

### 2.2 Data Cleaning 

Before any feature engineering or modelling could begin, the raw order book and trade data needed to be assessed for the kind of structural issues common in high-frequency financial data: irregular sampling, sparse trade activity, and incomplete joins between the two source tables. The data contains no sentinel or placeholder values, duplicate rows, or type errors requiring correction, and prices are pre-anonymised and normalised by Optiver, so no cleaning was needed on that front either; only relative price movement within a window is meaningful, not absolute price levels. The real work here was determining which apparent gaps in the data were genuine problems and which were expected features of how the data was collected, rather than assuming either by default. But first we must import some packages:

In [3]:
import duckdb
import pandas as pd
import numpy as np

In [4]:
con = duckdb.connect()

In [5]:
# missingness in seconds_in_bucket per window
con.sql("""
    SELECT stock_id, time_id, COUNT(*) AS n_updates,
           MIN(seconds_in_bucket) AS first_sec, MAX(seconds_in_bucket) AS last_sec
    FROM 'data/book_train.parquet'
    GROUP BY stock_id, time_id
    ORDER BY n_updates ASC
    LIMIT 10
""").df()

,stock_id,time_id,n_updates,first_sec,last_sec
0,18,29793,24,0,568
1,37,15852,33,0,545
2,33,12287,36,0,526
3,37,9518,39,0,590
4,33,15894,40,0,575
5,83,3552,40,0,585
6,37,11541,40,0,575
7,33,5658,42,0,505
8,33,32038,43,0,566
9,33,9285,44,0,585


In [ ]:
# any time_id/stock_id in train.csv missing from book data?
con.sql("""
    SELECT COUNT(*) FROM 'data/train.csv' t
    LEFT JOIN (SELECT DISTINCT stock_id, time_id FROM 'data/book_train.parquet') b
    ON t.stock_id = b.stock_id AND t.time_id = b.time_id
    WHERE b.stock_id IS NULL
""").df()

,count_star()
0,0


In [6]:
con.sql("""
    SELECT COUNT(*) AS total_windows,
           SUM(CASE WHEN n_trades = 0 THEN 1 ELSE 0 END) AS windows_with_no_trades,
           ROUND(100.0 * SUM(CASE WHEN n_trades = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_no_trades
    FROM (
        SELECT b.stock_id, b.time_id, COUNT(t.price) AS n_trades
        FROM (SELECT DISTINCT stock_id, time_id FROM 'data/book_train.parquet') b
        LEFT JOIN 'data/trade_train.parquet' t
        ON b.stock_id = t.stock_id AND b.time_id = t.time_id
        GROUP BY b.stock_id, b.time_id
    )
""").df()

,total_windows,windows_with_no_trades,pct_no_trades
0,428932,19.0,0.0


Three checks were run directly against the raw data using SQL:

- Irregular sampling. The order book is only recorded when it changes, not at fixed one-second intervals, so seconds_in_bucket contains gaps rather than a continuous sequence within each 10-minute window. In the sparsest windows, as few as 24 updates were recorded across the full window, with the last recorded update sometimes landing well before the 599-second mark. Rather than treating these gaps as missing data, the book state was forward-filled from the most recent update to reconstruct a continuous per-second price series where one was needed.

- Sparse trade activity. Trading was expected to occur less frequently than book updates, so a check was run for stock_id/time_id windows with no executed trades at all. Of 428,932 windows, only 19 (0.0044%) had zero recorded trades, meaning trade-derived features can be computed for effectively the entire dataset without extensive missingness handling, a considerably more complete picture than initially expected.

- Join integrity. Every stock_id/time_id combination in train.csv has matching records in the order book data, with zero unmatched rows confirmed via a left join.

Taken together, these checks show the dataset required structural handling, specifically accounting for irregular timestamps, rather than correction of genuine data errors. With sampling gaps addressed and both join coverage and trade activity confirmed to be effectively complete, the dataset was in a reliable state to move into feature engineering, with confidence that any missing values encountered downstream reflect real modelling decisions rather than artefacts of the raw data itself.


## 3. Methodology

This section describes the full pipeline used to go from raw order book and trade data to a trained, interpretable volatility prediction model. The approach follows a deliberate progression: first constructing the prediction target and establishing a naive baseline to serve as an honest floor, then engineering features from the raw data using SQL, before training and comparing models of increasing complexity. Feature engineering and target construction are done using DuckDB and SQL directly against the raw Parquet files, while model training and interpretability analysis are carried out in Python. Each stage is described in turn below, along with the reasoning behind key decisions such as the choice of evaluation metric and the baseline used for comparison.

### 3.1 Feature Engineering

The raw order book and trade data are recorded at irregular, sub-window granularity, hundreds of updates per stock_id/time_id, so before any model can be trained, this needs to be reduced to a single feature row per window. This section builds that feature table using SQL directly against the raw Parquet files: aggregating the order book into measures of spread, price volatility, and order size imbalance, and the trade data into measures of trading activity and volume. These features are combined via a left join, since not every window has trade activity, and the earlier data quality check confirmed this affects a genuinely small fraction of windows.


In [7]:
features = con.sql("""
    WITH wap_calc AS (
        SELECT
            stock_id, time_id, seconds_in_bucket,
            (bid_price1 * ask_size1 + ask_price1 * bid_size1) / (bid_size1 + ask_size1) AS wap,
            bid_price1, ask_price1, bid_size1, ask_size1
        FROM 'data/book_train.parquet'
    ),
    wap_returns AS (
        SELECT
            *,
            LN(wap) - LN(LAG(wap) OVER (PARTITION BY stock_id, time_id ORDER BY seconds_in_bucket)) AS log_return
        FROM wap_calc
    ),
    book_features AS (
        SELECT
            stock_id,
            time_id,
            AVG(ask_price1 - bid_price1) AS avg_spread,
            AVG(wap) AS avg_wap,
            STDDEV(log_return) AS wap_volatility,
            AVG((bid_size1 - ask_size1) * 1.0 / (bid_size1 + ask_size1)) AS avg_size_imbalance,
            MAX(ask_price1) - MIN(bid_price1) AS price_range,
            COUNT(*) AS n_book_updates
        FROM wap_returns
        GROUP BY stock_id, time_id
    ),
    trade_features AS (
        SELECT
            stock_id, time_id,
            COUNT(*) AS n_trades,
            SUM(size) AS total_trade_volume,
            AVG(size) AS avg_trade_size,
            SUM(order_count) AS total_order_count
        FROM 'data/trade_train.parquet'
        GROUP BY stock_id, time_id
    )
    SELECT
        b.*,
        COALESCE(t.n_trades, 0) AS n_trades,
        COALESCE(t.total_trade_volume, 0) AS total_trade_volume,
        COALESCE(t.avg_trade_size, 0) AS avg_trade_size,
        COALESCE(t.total_order_count, 0) AS total_order_count
    FROM book_features b
    LEFT JOIN trade_features t
    ON b.stock_id = t.stock_id AND b.time_id = t.time_id
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(428932, 12)


,stock_id,time_id,avg_spread,avg_wap,wap_volatility,avg_size_imbalance,price_range,n_book_updates,n_trades,total_trade_volume,avg_trade_size,total_order_count
0,110,4649,0.000389,0.998592,0.000093,-0.529429,0.002214,258,30,854.0,28.466667,42.0
1,110,5218,0.000828,1.002008,0.000214,-0.102323,0.004227,291,36,3544.0,98.444444,89.0
2,110,5676,0.000419,1.000217,0.000110,0.002441,0.002826,583,51,6508.0,127.607843,93.0
3,110,8825,0.000999,0.999568,0.000168,-0.238055,0.002575,205,11,1946.0,176.909091,37.0
4,110,12666,0.000843,0.999158,0.000198,-0.290652,0.004456,344,43,5135.0,119.418605,115.0


The resulting feature table contains 428,932 rows and 12 columns, one row per stock_id/time_id window. This row count matches exactly the number of unique windows identified in the earlier trade-sparsity check, confirming that the left join against the trade data did not drop or duplicate any windows.

The feature values are consistent with expectations for normalised order book data. avg_wap sits close to 1.0 across the sampled rows (0.9986–1.0020), reflecting Optiver's pre-normalised price scale. avg_spread and wap_volatility are both small positive values in the expected range, with no negative or anomalously large figures that would indicate an error in the WAP or log-return calculation. avg_size_imbalance varies naturally in both sign and magnitude (from -0.53 to 0.002 in the sample shown), correctly capturing windows with bid-side pressure, ask-side pressure, and near-balanced order flow, rather than being skewed toward one direction by a calculation error. n_trades is non-zero across all sampled rows, consistent with the earlier finding that only 0.0044% of windows have no trade activity, meaning the COALESCE(..., 0) handling affects a genuinely small fraction of the full table.

With one clean feature row per window confirmed, and no evidence of dropped data or calculation errors, this feature table is ready to be merged with the target values from train.csv ahead of baseline and model training in the following sections.

### 3.2 Target Construction 

The value being predicted, the realized volatility of the 10-minute window immediately following the one covered by the order book and trade data, is provided directly in train.csv rather than needing to be derived from scratch. Realized volatility itself is defined as the square root of the sum of squared log returns of a price series, the same formula already used to compute wap_volatility in Section 3.1, applied here to a subsequent window not included in the raw book data provided. This is precisely why the task is a genuine prediction problem rather than a calculation exercise: the features available describe what happened in one window, while the target describes what happens in the window after it, which cannot be observed directly. This section merges the engineered feature table with the target values and checks that the merge is clean before any modelling begins.

In [8]:
target = con.sql("SELECT * FROM 'data/train.csv' LIMIT 5").df()
target.head()

,stock_id,time_id,target
0,0,5,0.004136
1,0,11,0.001445
2,0,16,0.002168
3,0,31,0.002195
4,0,62,0.001747


In [9]:
model_data = features.merge(
    con.sql("SELECT * FROM 'data/train.csv'").df(),
    on=['stock_id', 'time_id'],
    how='inner'
)

print("Features shape:", features.shape)
print("Merged shape:", model_data.shape)
print("Any missing targets after merge:", model_data['target'].isnull().sum())

model_data[['stock_id', 'time_id', 'wap_volatility', 'target']].head()

Features shape: (428932, 12)
Merged shape: (428932, 13)
Any missing targets after merge: 0


,stock_id,time_id,wap_volatility,target
0,110,4649,0.000093,0.002066
1,110,5218,0.000214,0.003306
2,110,5676,0.000110,0.003046
3,110,8825,0.000168,0.002231
4,110,12666,0.000198,0.004733


The merge is clean: features and model_data both contain 428,932 rows, confirming no windows were dropped or duplicated, and model_data['target'].isnull().sum() returns 0, meaning every feature row has a corresponding target value. model_data is now ready to serve as the modelling dataset for the remainder of this report.

Looking at wap_volatility (current window) against target (next window) in the sample above, one thing stands out that's worth correcting from the framing in this section's introduction: wap_volatility is consistently smaller than target by roughly an order of magnitude (e.g. 0.000093 vs. 0.002066), rather than sitting on a directly comparable scale. This is because wap_volatility was computed using SQL's STDDEV(), which centres returns around their mean and normalises by n-1, whereas realized volatility as Optiver defines it, and as target represents, is the square root of the sum of squared log returns, with no normalisation by count and no mean-centring. The two are closely related statistically, since both measure dispersion in returns, but they are not the same formula, and wap_volatility should be understood as a correlated engineered feature rather than a same-window estimate of the target itself. This distinction matters for a feature used in prediction, but doesn't undermine its usefulness, and is worth flagging directly rather than implying equivalence, which the introduction to this section overstated.

With a clean, fully-populated modelling dataset in place, the next step is establishing a naive baseline before any real model is introduced.

### 3.3 Baseline Model 

Before training any real model, it's worth establishing a naive baseline: predicting each window's realized volatility using nothing more than the realized volatility of the window immediately preceding it. This exploits volatility clustering, the well-documented tendency for periods of high or low volatility to persist over short horizons, and gives an honest floor that any real model needs to clear to be worth using. Note that this uses the correctly-defined realized volatility formula (the square root of the sum of squared log returns), rather than reusing wap_volatility from Section 3.1, since Section 3.2 established that wap_volatility sits on a different scale due to its use of STDDEV(). Using it directly here would produce a misleadingly poor baseline purely from a scale mismatch, not from the persistence assumption actually failing. Evaluation uses RMSPE (Root Mean Squared Percentage Error), the metric used throughout this project and detailed further in Section 3.5.

In [10]:
current_vol = con.sql("""
    WITH wap_calc AS (
        SELECT
            stock_id, time_id, seconds_in_bucket,
            (bid_price1 * ask_size1 + ask_price1 * bid_size1) / (bid_size1 + ask_size1) AS wap
        FROM 'data/book_train.parquet'
    ),
    wap_returns AS (
        SELECT
            *,
            LN(wap) - LN(LAG(wap) OVER (PARTITION BY stock_id, time_id ORDER BY seconds_in_bucket)) AS log_return
        FROM wap_calc
    )
    SELECT
        stock_id, time_id,
        SQRT(SUM(POWER(log_return, 2))) AS realized_vol
    FROM wap_returns
    WHERE log_return IS NOT NULL
    GROUP BY stock_id, time_id
""").df()

baseline_data = current_vol.merge(
    con.sql("SELECT * FROM 'data/train.csv'").df(),
    on=['stock_id', 'time_id'],
    how='inner'
)

print(baseline_data.shape)
baseline_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(428932, 4)


,stock_id,time_id,realized_vol,target
0,31,15906,0.006207,0.003636
1,31,17387,0.002279,0.001652
2,31,20017,0.000368,0.002029
3,31,21145,0.005122,0.005738
4,31,22106,0.008615,0.013189


In [11]:
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

baseline_rmspe = rmspe(baseline_data['target'], baseline_data['realized_vol'])
print(f"Naive persistence baseline RMSPE: {baseline_rmspe:.4f}")

Naive persistence baseline RMSPE: 0.3414


The baseline dataset matches the expected 428,932 rows, confirming the merge preserved every window. The naive persistence baseline achieves an RMSPE of 0.3414.

Looking at the sample rows, the persistence assumption clearly has some signal, realized_vol and target move in the same general direction and rough magnitude across all five rows shown, rather than being unrelated. But the errors are also substantial in places: row 4 shows a current-window volatility of 0.0086 against a next-window target of 0.0132, a 53% relative miss, and row 2 shows the reverse pattern, undershooting by a similar margin. This is consistent with what an RMSPE of 0.34 represents: on average, the naive prediction is off by roughly a third of the true value in percentage terms, meaningful predictive signal, but far from precise.

This gives a concrete, honest floor for the modelling sections that follow. Any model trained in Section 3.4 needs to meaningfully beat 0.3414 to justify its added complexity over simply assuming volatility persists unchanged from one window to the next; a model that only marginally improves on this baseline would suggest the engineered features in Section 3.1 add little beyond what the current window's volatility already captures.

### 3.4 Main Model

With a naive persistence baseline established at 0.3414 RMSPE, this section trains two real models on the engineered feature set from Section 3.1: a linear regression as an intermediate step, and a LightGBM gradient boosting model as the main model. Both are evaluated on a held-out test split using the same RMSPE metric as the baseline, so results are directly comparable. One methodological detail matters here: LightGBM's default training objective minimises plain squared error, not percentage error, which doesn't match the metric this project actually cares about (as established in the RMSPE discussion above). To correct for this, each training example is weighted by 1/target², a standard technique for this exact problem that effectively turns a weighted squared-error loss into a proxy for RMSPE during training, rather than only using RMSPE at evaluation time.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import lightgbm as lgb

feature_cols = ['avg_spread', 'avg_wap', 'wap_volatility', 'avg_size_imbalance',
                 'price_range', 'n_book_updates', 'n_trades', 'total_trade_volume',
                 'avg_trade_size', 'total_order_count', 'stock_id']

X = model_data[feature_cols]
y = model_data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_rmspe = rmspe(y_test, lr_preds)
print(f"Linear Regression RMSPE: {lr_rmspe:.4f}")

Linear Regression RMSPE: 0.3933


In [16]:
# LightGBM, weighted to approximate RMSPE during training
weights_train = 1 / (y_train ** 2)

lgb_train = lgb.Dataset(X_train, label=y_train, weight=weights_train, categorical_feature=['stock_id'])
lgb_valid = lgb.Dataset(X_test, label=y_test, reference=lgb_train, categorical_feature=['stock_id'])

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_valid],
    callbacks=[lgb.early_stopping(stopping_rounds=20)]
)

lgb_preds = model.predict(X_test, num_iteration=model.best_iteration)
lgb_rmspe = rmspe(y_test, lgb_preds)

print(f"\nBaseline RMSPE:          {baseline_rmspe:.4f}")
print(f"Linear Regression RMSPE: {lr_rmspe:.4f}")
print(f"LightGBM RMSPE:          {lgb_rmspe:.4f}")

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's rmse: 0.00135199

Baseline RMSPE:          0.3414
Linear Regression RMSPE: 0.3933
LightGBM RMSPE:          0.2321


LightGBM substantially outperforms both the baseline and linear regression, achieving an RMSPE of 0.2321 against the naive persistence baseline of 0.3414, a 32% relative improvement. This places the model in a broadly competitive range with published solutions to this exact competition, which typically score between 0.20 and 0.23 RMSPE, suggesting the engineered features and weighted training objective are capturing genuine predictive signal rather than only marginal gains over persistence.

The linear regression result, however, is a more interesting finding than it first appears: at 0.3933 RMSPE, it performs worse than the naive baseline of 0.3414, meaning a simple linear model is actually less useful here than just assuming volatility persists unchanged from one window to the next. This is worth explaining rather than treating as a minor underperformance. Two likely causes:

- stock_id was included as a raw numeric feature, not a categorical one, in the linear model. LightGBM was explicitly told stock_id is categorical and can split on it accordingly, but scikit-learn's LinearRegression has no such distinction, it treats stock_id as an ordinary continuous number, implicitly assuming stock 80 is "twice" stock 40 in some meaningful sense, which is meaningless for an identifier. This likely distorts the fitted coefficients.

- The 1/target² sample weighting used to approximate RMSPE during LightGBM training was never applied to the linear regression fit. The linear model was trained to minimise plain squared error, a different objective than the one it's being evaluated on, while LightGBM's training objective was deliberately aligned with the evaluation metric.

### 3.5 Evaluation Metric 

Every model in this report, the naive baseline, linear regression, and LightGBM, was evaluated using RMSPE (Root Mean Squared Percentage Error), rather than a more conventional metric like RMSE or MAE. This choice was necessary rather than incidental: realized volatility varies substantially across the 112 stocks in this dataset, from calm, low-volatility names to more turbulent ones, and an absolute error metric would let high-volatility stocks dominate the score regardless of how well or poorly the model performed on calmer ones. RMSPE avoids this by measuring each prediction's error as a percentage of its true value, so a poor prediction counts as poor consistently, whatever the underlying stock's typical volatility level. This is also the metric used to score submissions in the original Optiver competition, so using it here keeps this project's results directly comparable to published benchmarks.

In [17]:
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

The formula squares each relative error before averaging and then takes the square root, meaning large relative misses are penalised disproportionately more than an equivalent number of small ones, a deliberate property given that a single badly-missed prediction is a more serious failure than several minor ones spread across different stocks.

This choice of metric directly shaped a modelling decision made in Section 3.4: since LightGBM's default training objective minimises plain squared error rather than percentage error, each training example was weighted by 1/target² during training. This effectively steers the model's internal optimisation toward the same relative-error criterion it is ultimately evaluated on, rather than leaving a mismatch between what the model is trained to minimise and what the final results are judged by. The linear regression model, notably, did not receive this same weighting, a gap identified in Section 3.4 as a likely contributor to its underperformance relative to even the naive baseline.

### 3.6 Interpretability

With LightGBM established as the strongest model, this section uses SHAP (SHapley Additive exPlanations) to understand why it makes the predictions it does, rather than treating it as a black box that simply outputs a lower RMSPE. SHAP values decompose each individual prediction into contributions from each feature, showing not just which features matter most on average, but the direction and magnitude of their effect on specific predictions. This matters for a project like this one: a model that predicts volatility well is only genuinely useful if its reasoning aligns with real market microstructure intuition, features like spread and order imbalance driving predictions in sensible directions, rather than exploiting some artefact of the data that happens to correlate with the target.

In [21]:
pip install matplotlib

     |████████████████████████████████| 7.8 MB 6.0 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 3.4 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 4.0 MB/s eta 0:00:01
     |████████████████████████████████| 2.9 MB 4.1 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 5.7 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 2.9 MB/s eta 0:00:01
You should consider upgrading via the '/Users/haydenwongg/data_science_project/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [23]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, feature_names=feature_cols)

ImportError: matplotlib is not installed so plotting is not available! Run `pip install matplotlib` to fix this.

In [18]:
shap_importance = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

shap_importance

NameError: name 'shap_values' is not defined